# Stage 3 — A 1D convolutional network (CNN) over time

**Goal:** give the network a sense of *time*. Stage 2 showed that an MLP fed the raw clip (flattened) reaches only about 45 %,
while the same MLP fed hand-made summaries reaches 53 % (5-fold CV). A 1D-CNN reads the raw clip frame by frame with small
sliding filters, so it can discover movement patterns by itself.

**Bars to beat** (5-fold CV, 300 signs, chance 0.33 %): Stage 1 linear SVM **46.8 ± 3.1 %**, Stage 2 MLP on summaries **53.0 ± 3.7 %**,
Stage 2 MLP on the raw sequence about **45 %** (single split).

**What you will learn here**
| Concept | Where |
|---|---|
| 1D convolution: a small filter that slides along time and shares its weights | Cells 6 and 8 |
| Receptive field: how many frames one unit "sees" | Cell 8 |
| Batch normalization, max-pooling, global pooling | Cell 6 |
| Why a CNN has about 10x fewer parameters than the flattened MLP | Cell 8 |
| Tuning on validation only: kernel size, width, input channels | Cell 11 |
| Seeds, 5-fold cross-validation, paired comparison with Stage 2 | Cells 12–13 |
| Leakage re-test with a stronger model | Cell 14 |

**Input:** each clip is a table of 64 frames x 132 numbers (the same cached file as Stage 2: `sequences_subset_T64.npz`;
the notebook rebuilds it if it is missing). For a CNN the table is turned around: 132 *channels* x 64 *time steps*.

**How to run:** open a **new Colab notebook**, use a **GPU runtime**, and run the cells in order from Cell 2.
Rough times (estimates, not measured): 20-60 s per run on a GPU; the whole notebook about 20-40 minutes.
If a cell fails, send me the last lines of the error.

In [ ]:
# [setup] Cell 2 — configuration
from pathlib import Path
import json, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/slovo_project')
except ImportError:                      # running outside Colab
    BASE = Path('./slovo_project')

PROC, FIG, RES = BASE / 'processed', BASE / 'figures', BASE / 'results'
for d in (FIG, RES):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
FIXED_T = 64
N_FOLDS = 5
SWEEP_SEEDS = [0, 1]      # two seeds per configuration while tuning (a single run is too noisy)
FINAL_SEEDS = [0, 1, 2]
warnings.filterwarnings('ignore')
print('torch', torch.__version__, '| device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'no GPU')

In [ ]:
# Cell 3 — load the Stage 0 outputs (same files and same split as Stages 1 and 2)
splits = pd.read_csv(PROC / 'splits.csv')
z = np.load(PROC / 'landmarks_raw.npz', allow_pickle=False)
ids, lengths = z['ids'], z['lengths']
offsets = np.concatenate([[0], np.cumsum(lengths)])
id2idx = {c: i for i, c in enumerate(ids)}
frames = np.load(PROC / 'frames_tracked_v2.npy', mmap_mode='r')
assert len(frames) == offsets[-1], 'frames_tracked_v2.npy does not match landmarks_raw.npz: re-run Stage 0 cells 10 and 11b.'

meta = splits[splits['in_subset']].reset_index(drop=True)
y_all = meta['label_id_subset'].astype(int).to_numpy()
N_CLASSES = int(y_all.max()) + 1
assert len(np.unique(y_all)) == N_CLASSES

tr = (meta['split'] == 'train').to_numpy(); va = (meta['split'] == 'val').to_numpy(); te = (meta['split'] == 'test').to_numpy()
folds = meta['fold'].to_numpy()                                         # -1 = prolific signers (always train)
print(f'{len(meta)} clips | {N_CLASSES} signs | chance {1 / N_CLASSES:.2%} | train {tr.sum()} val {va.sum()} test {te.sum()}')

In [ ]:
# [sequence] Cell 4 — the sequence input: 64 frames x 132 numbers per clip (cached by Stage 2; rebuilt if missing)
from tqdm.auto import tqdm
WRIST, MIDDLE_MCP = 0, 9

def normalize_hands(x, aspect=None, eps=1e-6):
    """x: (..., 21, 3). Aspect-correct x, centre on the wrist, divide by the wrist->middle-finger-base distance."""
    x = x.astype(np.float32).copy()
    if aspect is not None:
        x[..., 0] *= aspect
    centred = x - x[..., WRIST:WRIST + 1, :]
    scale = np.linalg.norm(centred[..., MIDDLE_MCP, :2], axis=-1)
    scale = np.where(scale < eps, np.nan, scale)
    return centred / scale[..., None, None]

def clip_to_sequence(clip, aspect, T=FIXED_T):
    """(L, 2, 21, 3) -> (T, 132): 126 normalized joint values, 4 wrist-position values, 2 'hand visible' flags."""
    L = len(clip)
    idx = np.linspace(0, L - 1, T).round().astype(int)
    hands = normalize_hands(clip, aspect=aspect)[idx]
    wrist = clip[:, :, WRIST, :2].astype(np.float32).copy()
    wrist[..., 0] *= aspect
    wrist = wrist[idx]
    present = ~np.isnan(hands).all(axis=(2, 3))
    hands = np.where(np.isnan(hands), 0.0, hands)
    wrist = np.where(present[:, :, None], wrist, 0.0)
    return np.concatenate([hands.reshape(T, 126), wrist.reshape(T, 4), present.astype(np.float32)], axis=1).astype(np.float32)

SEQF = PROC / f'sequences_subset_T{FIXED_T}.npz'
meta_ids = meta['attachment_id'].to_numpy().astype(str)
SEQ = None
if SEQF.exists():
    zs = np.load(SEQF, allow_pickle=False)
    if zs['ids'].shape == meta_ids.shape and (zs['ids'] == meta_ids).all():
        SEQ = zs['S'].astype(np.float32)
        print('Loaded cached sequences:', SEQF.name)
if SEQ is None:
    SEQ = np.zeros((len(meta), FIXED_T, 132), np.float32)
    for r, (cid, asp) in enumerate(tqdm(zip(meta['attachment_id'], meta['aspect']), total=len(meta))):
        i = id2idx[cid]
        SEQ[r] = clip_to_sequence(np.asarray(frames[offsets[i]:offsets[i + 1]], dtype=np.float32), float(asp))
    np.savez_compressed(SEQF, S=SEQ.astype(np.float16), ids=meta_ids)
    print('Saved', SEQF.name)
assert SEQ.shape == (len(meta), FIXED_T, 132) and np.isfinite(SEQ).all()
print('sequence tensor:', SEQ.shape)

In [ ]:
# [prep] Cell 5 — normalization from TRAIN rows only, channel choices, and the (channels, time) layout a CNN expects
def seq_mask(S):
    """(N, T, 132) mask: 1 where the channel's hand is visible (or the channel is a visibility flag)."""
    p = S[:, :, 130:132]
    return np.concatenate([np.repeat(p[:, :, :1], 63, 2), np.repeat(p[:, :, 1:], 63, 2),
                           np.repeat(p[:, :, :1], 2, 2), np.repeat(p[:, :, 1:], 2, 2), np.ones_like(p)], axis=2)

def seq_stats(S_tr):
    m = seq_mask(S_tr)
    cnt = np.maximum(m.sum((0, 1)), 1.0)
    mean = (S_tr * m).sum((0, 1)) / cnt
    std = np.sqrt((((S_tr - mean) ** 2) * m).sum((0, 1)) / cnt)
    std = np.where(std < 1e-3, 1.0, std)
    mean[130:], std[130:] = 0.0, 1.0
    return mean.astype(np.float32), std.astype(np.float32)

def apply_seq_norm(S, mean, std):
    return (((S - mean) / std) * seq_mask(S)).astype(np.float32)           # missing hands stay exactly 0

# which of the 132 channels the network may use
INPUTS = {'all': np.arange(132),                                            # joints + wrist position + visibility
          'no_position': np.r_[0:126, 130:132],                             # joints + visibility (no wrist position)
          'joints_only': np.arange(126)}                                    # joints only

def prepare_cnn(input_name, train_mask, masks):
    """Normalize with TRAIN statistics, keep the chosen channels, return {name: (N, channels, time)} arrays."""
    mean, std = seq_stats(SEQ[train_mask])
    ch = INPUTS[input_name]
    return {k: np.ascontiguousarray(apply_seq_norm(SEQ[m], mean, std)[:, :, ch].transpose(0, 2, 1)) for k, m in masks.items()}

_d = prepare_cnn('all', tr, {'train': tr})['train']
assert _d.shape == (tr.sum(), 132, FIXED_T) and np.isfinite(_d).all()
assert prepare_cnn('no_position', tr, {'train': tr})['train'].shape[1] == 128
assert prepare_cnn('joints_only', tr, {'train': tr})['train'].shape[1] == 126
print('OK: CNN input layout (clips, channels, time) =', _d.shape)
del _d

In [ ]:
# [torch] Cell 6 — the 1D-CNN and the training loop (the loop is the same as in Stage 2)
import torch.nn as nn
import torch.nn.functional as F

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class CNN1D(nn.Module):
    """Blocks of [Conv1d -> BatchNorm -> ReLU -> MaxPool(2) -> Dropout], then global average + max pooling over time,
    then a Linear layer with one score per sign.  Input shape: (batch, channels, time)."""
    def __init__(self, in_ch, n_classes, channels=(128, 128, 256), kernel=5, dropout=0.5, block_dropout=0.1):
        super().__init__()
        blocks, c_in = [], in_ch
        for c_out in channels:
            blocks += [nn.Conv1d(c_in, c_out, kernel, padding=kernel // 2), nn.BatchNorm1d(c_out), nn.ReLU(),
                       nn.MaxPool1d(2), nn.Dropout(block_dropout)]
            c_in = c_out
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * c_in, n_classes))

    def forward(self, x):
        h = self.features(x)                                      # (batch, c, time / 2^blocks)
        h = torch.cat([h.mean(dim=2), h.amax(dim=2)], dim=1)      # global average + global max over time
        return self.head(h)

def new_model(in_ch, cfg, seed):
    set_seed(seed)
    return CNN1D(in_ch, N_CLASSES, cfg['channels'], cfg['kernel'], cfg['dropout'], cfg['block_dropout']).to(DEVICE)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

@torch.no_grad()
def check_architecture(in_ch, cfg):
    """Pass a dummy batch through the network: returns (output shape, number of parameters)."""
    m = new_model(in_ch, cfg, 0).eval()
    out = m(torch.zeros(2, in_ch, FIXED_T, device=DEVICE))
    return tuple(out.shape), count_params(m)

@torch.no_grad()
def _eval(model, X, y):
    model.eval()                                   # dropout off, batch-norm uses its running statistics
    logits = model(X)
    return F.cross_entropy(logits, y).item(), (logits.argmax(1) == y).float().mean().item()

def fit(model, Xtr, ytr, Xva=None, yva=None, epochs=150, lr=1e-3, weight_decay=1e-2, batch_size=128,
        patience=20, label_smoothing=0.0, seed=0, fixed_epochs=None):
    """Mini-batch training. With a validation set: stop when validation accuracy has not improved for `patience`
    epochs and restore the best weights. With fixed_epochs: train exactly that many epochs, no validation."""
    t0 = time.time()
    n_epochs = fixed_epochs or epochs
    g = torch.Generator(); g.manual_seed(seed)
    Xt = torch.as_tensor(Xtr, dtype=torch.float32, device=DEVICE)
    yt = torch.as_tensor(ytr, dtype=torch.long, device=DEVICE)
    use_val = (Xva is not None) and (fixed_epochs is None)
    if use_val:
        Xv = torch.as_tensor(Xva, dtype=torch.float32, device=DEVICE)
        yv = torch.as_tensor(yva, dtype=torch.long, device=DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    hist = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc, best_epoch, best_state, bad, n = -1.0, 0, None, 0, len(Xt)
    for epoch in range(1, n_epochs + 1):
        model.train()
        perm = torch.randperm(n, generator=g).to(DEVICE)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(Xt[idx]), yt[idx])
            loss.backward()
            opt.step()
        tl, ta = _eval(model, Xt, yt)
        hist['train_loss'].append(tl); hist['train_acc'].append(ta)
        if use_val:
            vl, vacc = _eval(model, Xv, yv)
            hist['val_loss'].append(vl); hist['val_acc'].append(vacc)
            if vacc > best_acc:
                best_acc, best_epoch, bad = vacc, epoch, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience:
                    break
    if use_val and best_state is not None:
        model.load_state_dict(best_state)
    else:
        best_epoch = n_epochs
    return hist, best_epoch, time.time() - t0

@torch.no_grad()
def predict_logits(model, X, batch_size=512):
    model.eval()
    out = []
    for i in range(0, len(X), batch_size):
        xb = torch.as_tensor(X[i:i + batch_size], dtype=torch.float32, device=DEVICE)
        out.append(model(xb).cpu().numpy())
    return np.concatenate(out)

print('CNN and training loop defined')

In [ ]:
# [runner] Cell 7 — metrics and the experiment runner
from sklearn.metrics import f1_score

def metrics_from_logits(logits, y):
    pred = logits.argmax(1)
    top5 = np.argsort(-logits, axis=1)[:, :5]
    return {'n': int(len(y)), 'top1': float((pred == y).mean()), 'top5': float(np.mean([yt in t for yt, t in zip(y, top5)])),
            'macro_f1': float(f1_score(y, pred, average='macro')), 'pred': pred}

DEFAULT_CFG = dict(channels=(128, 128, 256), kernel=5, dropout=0.5, block_dropout=0.1, lr=1e-3, weight_decay=1e-2,
                   batch_size=128, epochs=150, patience=20, label_smoothing=0.1, input='all')
MASKS = {'train': tr, 'val': va, 'test': te}
results = []                    # one dict per (experiment, run, evaluation set)
_cache, _data_cache = {}, {}    # finished runs and prepared inputs, so identical runs are never repeated

def mask_key(masks):
    return tuple((k, hash(m.tobytes())) for k, m in masks.items())          # identifies the exact set of clips

def get_data(input_name, masks, cache=True):
    if not cache:
        return prepare_cnn(input_name, masks['train'], masks)
    key = (input_name, mask_key(masks))
    if key not in _data_cache:
        _data_cache[key] = prepare_cnn(input_name, masks['train'], masks)
    return _data_cache[key]

def run_one(experiment, name, masks, seed, cfg=None, fixed_epochs=None, use_cache=True):
    cfg = {**DEFAULT_CFG, **(cfg or {})}
    ckey = (json.dumps(cfg, sort_keys=True, default=list), seed, mask_key(masks), fixed_epochs)
    if use_cache and ckey in _cache:
        return _cache[ckey]
    data = get_data(cfg['input'], masks, cache=use_cache)
    model = new_model(data['train'].shape[1], cfg, seed)
    hist, best_ep, fit_s = fit(model, data['train'], y_all[masks['train']],
                               data.get('val'), y_all[masks['val']] if 'val' in masks else None,
                               epochs=cfg['epochs'], lr=cfg['lr'], weight_decay=cfg['weight_decay'],
                               batch_size=cfg['batch_size'], patience=cfg['patience'],
                               label_smoothing=cfg['label_smoothing'], seed=seed, fixed_epochs=fixed_epochs)
    out = {'name': name, 'seed': seed, 'hist': hist, 'best_epoch': best_ep, 'fit_s': fit_s, 'metrics': {}}
    for sname, m in masks.items():
        if sname == 'train':
            continue
        r = metrics_from_logits(predict_logits(model, data[sname]), y_all[m])
        out['metrics'][sname] = r
        results.append({'experiment': experiment, 'name': name, 'seed': seed, 'set': sname, 'n': r['n'],
                        'top1': r['top1'], 'top5': r['top5'], 'macro_f1': r['macro_f1'], 'best_epoch': best_ep, 'fit_s': fit_s})
    if use_cache:
        _cache[ckey] = out
    return out

def summary_line(o, sets=('val', 'test')):
    return ' | '.join(f'{s} top-1 {o["metrics"][s]["top1"]:.1%} top-5 {o["metrics"][s]["top5"]:.1%}' for s in sets if s in o['metrics']) \
           + f' | best epoch {o["best_epoch"]} | {o["fit_s"]:.0f}s'

In [ ]:
# [arch] Cell 8 — what does the network look like? receptive field, shapes and parameter count
def receptive_field(kernel, n_blocks, pool=2):
    """How many input frames influence one output time step after n blocks of [conv(kernel) -> max-pool(pool)]."""
    rf, jump = 1, 1
    for _ in range(n_blocks):
        rf += (kernel - 1) * jump
        rf += (pool - 1) * jump
        jump *= pool
    return rf

assert receptive_field(5, 3) == 36 and receptive_field(3, 3) == 22

for k in (3, 5, 9):
    print(f'kernel {k}: after 3 blocks each time step sees {receptive_field(k, 3)} of the {FIXED_T} frames')

shape, n_params = check_architecture(132, DEFAULT_CFG)
assert shape == (2, N_CLASSES), f'unexpected output shape {shape}'
print(f'\nCNN output shape {shape} | parameters: {n_params:,}')
print(f'for comparison, the Stage 2 MLP on the flattened sequence has {(8448 * 512 + 512 * 256 + 256 * N_CLASSES):,} weights in its three Linear layers')

In [ ]:
# Cell 9 — sanity check: can the CNN memorize 64 training clips? (if not, something is wrong)
d = prepare_cnn('all', tr, {'train': tr})['train']
small = np.random.default_rng(0).choice(np.where(tr)[0], 64, replace=False)
sel = np.isin(np.where(tr)[0], small)
cfg0 = {**DEFAULT_CFG, 'dropout': 0.0, 'block_dropout': 0.0}
mdl = new_model(d.shape[1], cfg0, 0)
h, _, _ = fit(mdl, d[sel], y_all[tr][sel], fixed_epochs=200, lr=3e-3, weight_decay=0.0, batch_size=64, seed=0)
print(f'train loss {h["train_loss"][0]:.2f} -> {h["train_loss"][-1]:.3f} | train accuracy {h["train_acc"][0]:.0%} -> {h["train_acc"][-1]:.0%}')
assert h['train_acc'][-1] > 0.9, 'The CNN cannot even memorize 64 clips: something is wrong.'
print('OK: the CNN can learn')
del d, mdl

In [ ]:
# Cell 10 — Experiment 1: the baseline CNN on the raw sequence, and its learning curves
o1 = run_one('E1 baseline CNN', 'CNN baseline', MASKS, seed=0)
print(summary_line(o1), f'| final train acc {o1["hist"]["train_acc"][-1]:.0%}')

h = o1['hist']; ep = np.arange(1, len(h['train_loss']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ep, h['train_loss'], label='train'); ax[0].plot(ep, h['val_loss'], label='validation')
ax[0].axvline(o1['best_epoch'], color='k', ls=':', label='best epoch'); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend(); ax[0].set_title('Loss')
ax[1].plot(ep, h['train_acc'], label='train'); ax[1].plot(ep, h['val_acc'], label='validation')
ax[1].axhline(0.448, color='tab:red', ls='--', lw=1, label='Stage 2 MLP on sequence (val)')
ax[1].axhline(0.53, color='gray', ls='--', lw=1, label='Stage 2 MLP on features (CV)')
ax[1].set_xlabel('epoch'); ax[1].legend(fontsize=7); ax[1].set_title('Accuracy')
plt.tight_layout(); plt.savefig(FIG / 'stage3_e1_curves.png', dpi=150); plt.show()

In [ ]:
# Cell 11 — Experiment 2: tune three design choices on VALIDATION only (greedy, two seeds per option)
SWEEPS = [('kernel', 'kernel size (frames seen by one filter)', [3, 5, 9]),
          ('channels', 'width (filters per block)', [(64, 64, 128), (128, 128, 256), (256, 256, 512)]),
          ('input', 'input channels', ['all', 'no_position', 'joints_only'])]
best_cfg = dict(DEFAULT_CFG)
sweep_rows = []
for key, label, options in SWEEPS:
    scored = []
    for opt in options:
        cfg = {**best_cfg, key: opt}
        outs = [run_one(f'E2 sweep {key}', f'{key}={opt}', MASKS, seed=s, cfg=cfg) for s in SWEEP_SEEDS]
        v = [o['metrics']['val']['top1'] for o in outs]
        n_par = check_architecture(len(INPUTS[cfg['input']]), cfg)[1]
        scored.append((np.mean(v), opt))
        sweep_rows.append({'choice': key, 'option': str(opt), 'val_top1_mean': np.mean(v), 'val_top1_std': np.std(v),
                           'params': n_par, 'best_epochs': [o['best_epoch'] for o in outs]})
        print(f'{label:42s} {str(opt):18s} val top-1 {np.mean(v):.1%} +/- {np.std(v):.1%} | {n_par:>9,} parameters')
    best = max(scored, key=lambda t: t[0])
    best_cfg[key] = best[1]
    print(f'  -> chosen: {key} = {best[1]}\n')
print('final configuration:', {k: best_cfg[k] for k in ('kernel', 'channels', 'input')})
pd.DataFrame(sweep_rows).to_csv(RES / 'stage3_sweeps.csv', index=False)

In [ ]:
# Cell 12 — Experiment 3: the chosen CNN with three seeds, compared with Stage 2
finals = [run_one('E3 seeds', 'CNN tuned', MASKS, seed=s, cfg=best_cfg, use_cache=False) for s in FINAL_SEEDS]
v = [o['metrics']['val']['top1'] for o in finals]; t = [o['metrics']['test']['top1'] for o in finals]; t5 = [o['metrics']['test']['top5'] for o in finals]
print(f'CNN  val top-1 {np.mean(v):.1%} +/- {np.std(v):.1%} | test top-1 {np.mean(t):.1%} +/- {np.std(t):.1%} | test top-5 {np.mean(t5):.1%}'
      f' | per-seed test: ' + ', '.join(f'{x:.1%}' for x in t))
BEST_EPOCH = int(np.median([o['best_epoch'] for o in finals]))

s2_path = RES / 'stage2_results.csv'
if s2_path.exists():
    s2 = pd.read_csv(s2_path)
    for nm in ('MLP on features', 'MLP on sequence'):
        r = s2[(s2.experiment == 'E4 seeds') & (s2.name == nm) & (s2.set == 'test')].top1
        if len(r):
            print(f'Stage 2 {nm:18s} test top-1 {r.mean():.1%} +/- {r.std(ddof=0):.1%}  (3 seeds)')
else:
    print('(results/stage2_results.csv not found: Stage 2 test top-1 was 57.3% for the features MLP and 45.2% for the sequence MLP)')

In [ ]:
# Cell 13 — Experiment 4: 5-fold cross-validation (same folds as Stages 1 and 2) and a paired comparison
cv_acc, pooled_y, pooled_p = [], [], []
for k in range(N_FOLDS):
    te_k = folds == k; va_k = folds == (k + 1) % N_FOLDS; tr_k = ~(te_k | va_k)
    o = run_one('E4 5-fold CV', 'CNN tuned', {'train': tr_k, 'val': va_k, 'test': te_k}, seed=0, cfg=best_cfg, use_cache=False)
    results[-2]['set'], results[-1]['set'] = f'fold {k} val', f'fold {k}'
    cv_acc.append(o['metrics']['test']['top1']); pooled_y.append(y_all[te_k]); pooled_p.append(o['metrics']['test']['pred'])
    print(f'fold {k}: test clips {te_k.sum()} | top-1 {cv_acc[-1]:.1%} | best epoch {o["best_epoch"]} | {o["fit_s"]:.0f}s')
pooled_y, pooled_p = np.concatenate(pooled_y), np.concatenate(pooled_p)
cv_acc = np.array(cv_acc)
print(f'\nCNN: top-1 mean {cv_acc.mean():.1%} +/- {cv_acc.std():.1%} over folds | pooled {np.mean(pooled_y == pooled_p):.1%} on {len(pooled_y)} clips\n')

def cv_from_csv(path, experiment, name):
    if not path.exists():
        return None
    d = pd.read_csv(path)
    d = d[(d.experiment == experiment) & (d.name == name) & d.set.str.fullmatch(r'fold \d')] if 'name' in d else \
        d[(d.experiment == experiment) & (d.model == name) & d.set.str.fullmatch(r'fold \d')]
    v = d.sort_values('set').top1.to_numpy()
    return v if len(v) == N_FOLDS else None

refs = {'Stage 1 linear SVM': cv_from_csv(RES / 'stage1_results.csv', '5-fold CV', 'linear SVM'),
        'Stage 2 MLP on features': cv_from_csv(RES / 'stage2_results.csv', 'E5 5-fold CV', 'MLP on features')}
for nm, v in refs.items():
    if v is None:
        print(f'{nm}: CSV not found'); continue
    diff = cv_acc - v
    print(f'{nm:26s} {v.mean():.1%} +/- {v.std():.1%} | CNN better in {(diff > 0).sum()} of {N_FOLDS} folds | mean difference {diff.mean():+.1%} (per fold: '
          + ', '.join(f'{x:+.1%}' for x in diff) + ')')

In [ ]:
# Cell 14 — Experiment 5: signer leakage with the CNN (official split: seen vs unseen signers)
off_train = meta['official_train'].to_numpy(bool)
unseen = meta['official_test_unseen_signer'].to_numpy(bool)
seen = (~off_train) & (~unseen)
print(f'official train {off_train.sum()} | official test: {seen.sum()} clips from seen signers, {unseen.sum()} from unseen signers | fixed epochs {BEST_EPOCH}')
leak = [run_one('E5 leakage', 'CNN tuned', {'train': off_train, 'seen': seen, 'unseen': unseen}, seed=s, cfg=best_cfg,
                fixed_epochs=BEST_EPOCH, use_cache=False) for s in FINAL_SEEDS]
a = [o['metrics']['seen']['top1'] for o in leak]; b = [o['metrics']['unseen']['top1'] for o in leak]
print(f'seen signers {np.mean(a):.1%} +/- {np.std(a):.1%} | unseen signers {np.mean(b):.1%} +/- {np.std(b):.1%} | gap {np.mean(a) - np.mean(b):+.1%}')
print('(Stage 1 logistic regression: gap +0.5; Stage 2 MLP on features: seen 49.6%, unseen 46.7%, gap +2.9; '
      'the standard error of a gap with these test sets is about 2.7 points)')

In [ ]:
# Cell 15 — results table and the final figure
res = pd.DataFrame(results)
res.to_csv(RES / 'stage3_results.csv', index=False)
print(res.groupby(['experiment', 'name', 'set'])[['top1', 'top5', 'macro_f1']].mean().round(3).to_string())

labels, means, stds = [], [], []
for nm, v in refs.items():
    if v is not None:
        labels.append(nm); means.append(v.mean()); stds.append(v.std())
labels.append('Stage 3: CNN on sequence'); means.append(cv_acc.mean()); stds.append(cv_acc.std())
plt.figure(figsize=(8, 3.8))
plt.bar(range(len(labels)), means, yerr=stds, capsize=4, color=['#999'] * (len(labels) - 1) + ['#4C78A8'])
plt.xticks(range(len(labels)), labels, rotation=20, ha='right'); plt.ylabel('top-1 accuracy (5-fold CV, mean +/- std)')
plt.axhline(1 / N_CLASSES, color='k', ls='--', lw=1); plt.title('Bars to beat vs the first CNN')
plt.tight_layout(); plt.savefig(FIG / 'stage3_results.png', dpi=150); plt.show()
print('Saved', RES / 'stage3_results.csv', RES / 'stage3_sweeps.csv', 'and', FIG / 'stage3_results.png')

## Stage 3 checklist

- [ ] Cell 8: receptive fields and parameter count printed, no assertion error
- [ ] Cell 9 printed `OK: the CNN can learn`
- [ ] Cell 10: learning curves with the two reference lines
- [ ] Cell 11: the three tuning tables and the final configuration
- [ ] Cell 12: 3 seeds compared with Stage 2
- [ ] Cell 13: 5-fold CV and the paired comparison
- [ ] `results/stage3_results.csv`, `results/stage3_sweeps.csv` and the figures exist

**Send back:** the printed outputs of Cells 8 to 14, and the figures `stage3_e1_curves.png` and `stage3_results.png`.
If a cell fails, send me the last lines of the error.